In [1]:
!pip install -q datasets indic-nlp-library pandas numpy tqdm sentencepiece tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 11.0 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from indicnlp.tokenize import sentence_tokenize

from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

from tqdm import tqdm
import pandas as pd
import numpy as np
import random
import os

In [3]:
from huggingface_hub import login

login()

In [4]:
dataset_dict = load_dataset(
    "ai4bharat/IndicCorpV2",
    streaming=True
)

languages = list(dataset_dict.keys())

print("Number of languages:", len(languages))
print("Available languages:")
print(languages)

README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

Number of languages: 23
Available languages:
['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab', 'khasi', 'santhali']


In [5]:
LANGUAGE = "hin_Deva"

dataset = dataset_dict[LANGUAGE]

print("Selected language:", LANGUAGE)
print("Dataset type:", type(dataset))

Selected language: hin_Deva
Dataset type: <class 'datasets.iterable_dataset.IterableDataset'>


In [7]:
dataset = dataset_dict["hin_Deva"]

In [8]:
for row in dataset:
    print("Available columns:", row.keys())
    print("\nSample paragraph:\n")
    print(row["text"][:2000])
    break

Available columns: dict_keys(['text'])

Sample paragraph:

लोगों को बिलों संबंधी सुविधा देना ही उनका काम


In [9]:
TARGET_SENTENCES = 1_200_000

all_sentences = []

for row in tqdm(dataset, desc="Reading paragraphs"):

    text = row["text"]

    try:
        sentences = sentence_tokenize.sentence_split(
            text,
            lang="hi"
        )

        for sentence in sentences:

            sentence = sentence.strip()

            # Remove very short/noisy sentences
            if len(sentence.split()) >= 3:
                all_sentences.append(sentence)

        if len(all_sentences) >= TARGET_SENTENCES:
            break

    except Exception:
        continue

print("Total sentences collected:", len(all_sentences))

Reading paragraphs: 763192it [00:28, 26541.79it/s]

Total sentences collected: 1200003


In [10]:
all_sentences = list(dict.fromkeys(all_sentences))

print("Unique sentences:", len(all_sentences))

if len(all_sentences) < 1_002_000:
    print("WARNING: Not enough sentences.")
else:
    print("Enough sentences available.")

Unique sentences: 1193558
Enough sentences available.


In [11]:
sentence_lengths = np.array([
    len(sentence.split())
    for sentence in all_sentences
])

print("Minimum length:", sentence_lengths.min())
print("Maximum length:", sentence_lengths.max())
print("Average length:", sentence_lengths.mean())
print("Median length:", np.median(sentence_lengths))

Minimum length: 3
Maximum length: 1682
Average length: 18.37539943597211
Median length: 16.0


In [12]:
bins = [0, 5, 10, 15, 20, 30, 40, 60, 100, np.inf]

length_bins = pd.cut(
    sentence_lengths,
    bins=bins,
    right=True
)

print(length_bins.value_counts().sort_index())

(0.0, 5.0]        39542
(5.0, 10.0]      229912
(10.0, 15.0]     306340
(15.0, 20.0]     243377
(20.0, 30.0]     250944
(30.0, 40.0]      81251
(40.0, 60.0]      33720
(60.0, 100.0]      6081
(100.0, inf]       2391
Name: count, dtype: int64


In [13]:
def stratified_sample(sentences, n, seed=42):

    random.seed(seed)

    lengths = np.array([
        len(sentence.split())
        for sentence in sentences
    ])

    bins = [0, 5, 10, 15, 20, 30, 40, 60, 100, np.inf]

    categories = pd.cut(
        lengths,
        bins=bins,
        right=True
    )

    df = pd.DataFrame({
        "sentence": sentences,
        "category": categories
    })

    selected_parts = []

    # Proportion of each length category
    proportions = df["category"].value_counts(
        normalize=True,
        sort=False
    )

    for category, proportion in proportions.items():

        group = df[df["category"] == category]

        count = max(
            1,
            int(round(proportion * n))
        )

        count = min(count, len(group))

        selected = group.sample(
            n=count,
            random_state=seed
        )

        selected_parts.append(selected)

    result = pd.concat(selected_parts)

    # If slightly more than required
    if len(result) > n:
        result = result.sample(
            n=n,
            random_state=seed
        )

    # If slightly less than required
    elif len(result) < n:

        remaining = df.drop(
            result.index,
            errors="ignore"
        )

        extra = remaining.sample(
            n=n - len(result),
            random_state=seed
        )

        result = pd.concat([
            result,
            extra
        ])

    result = result.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    return result["sentence"].tolist()

In [14]:
DEV_SIZE = 1000

dev_sentences = stratified_sample(
    all_sentences,
    DEV_SIZE,
    seed=42
)

print("Development sentences:", len(dev_sentences))

Development sentences: 1000


In [15]:
dev_set = set(dev_sentences)

remaining_sentences = [
    sentence
    for sentence in all_sentences
    if sentence not in dev_set
]

print("Remaining sentences:", len(remaining_sentences))

Remaining sentences: 1192558


In [16]:
TEST_SIZE = 1000

test_sentences = stratified_sample(
    remaining_sentences,
    TEST_SIZE,
    seed=123
)

print("Test sentences:", len(test_sentences))

Test sentences: 1000


In [17]:
dev_set = set(dev_sentences)
test_set = set(test_sentences)

print("Development size:", len(dev_set))
print("Test size:", len(test_set))

print("Overlap:", len(dev_set & test_set))

Development size: 1000
Test size: 1000
Overlap: 0


In [18]:
dev_lengths = [len(s.split()) for s in dev_sentences]
test_lengths = [len(s.split()) for s in test_sentences]

distribution = pd.DataFrame({
    "Statistic": [
        "Minimum",
        "Maximum",
        "Mean",
        "Median"
    ],
    "Development": [
        min(dev_lengths),
        max(dev_lengths),
        np.mean(dev_lengths),
        np.median(dev_lengths)
    ],
    "Test": [
        min(test_lengths),
        max(test_lengths),
        np.mean(test_lengths),
        np.median(test_lengths)
    ]
})

display(distribution)

,Statistic,Development,Test
0,Minimum,3.000,3.000
1,Maximum,280.000,235.000
2,Mean,18.266,18.324
3,Median,16.000,16.000


In [19]:
special_sentences = dev_set | test_set

training_pool = [
    sentence
    for sentence in all_sentences
    if sentence not in special_sentences
]

print("Training pool:", len(training_pool))
print("Development:", len(dev_sentences))
print("Test:", len(test_sentences))

Training pool: 1191558
Development: 1000
Test: 1000


In [20]:
TRAINING_SIZES = [
    100_000,
    300_000,
    500_000,
    1_000_000
]

training_sets = {}

random.seed(42)

shuffled_training = training_pool.copy()
random.shuffle(shuffled_training)

for size in TRAINING_SIZES:

    training_sets[size] = shuffled_training[:size]

    print(
        f"Training size {size:,}:",
        len(training_sets[size])
    )

Training size 100,000: 100000
Training size 300,000: 300000
Training size 500,000: 500000
Training size 1,000,000: 1000000


In [21]:
def train_bpe(sentences, vocab_size):

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"]
    )

    tokenizer.train_from_iterator(
        sentences,
        trainer=trainer
    )

    return tokenizer

In [22]:
def train_wordpiece(sentences, vocab_size):

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=[
            "[UNK]",
            "[PAD]",
            "[CLS]",
            "[SEP]"
        ]
    )

    tokenizer.train_from_iterator(
        sentences,
        trainer=trainer
    )

    return tokenizer

In [23]:
bpe_tokenizer = train_bpe(
    training_sets[100_000],
    vocab_size=20_000
)

print(
    "BPE vocabulary size:",
    bpe_tokenizer.get_vocab_size()
)

BPE vocabulary size: 20000


In [24]:
dev_encoding = bpe_tokenizer.encode(dev_sentences[0])
test_encoding = bpe_tokenizer.encode(test_sentences[0])

print("Original sentence:")
print(dev_sentences[0])

print("\nBPE tokens:")
print(dev_encoding.tokens)

print("\nToken IDs:")
print(dev_encoding.ids)

Original sentence:
आम तौर पर दो-चार साल पर इलाज में महंगाई के मुताबिक़ बीमा कंपनियां प्रीमियम में फेरबदल करती हैं.

BPE tokens:
['आम', 'तौर', 'पर', 'दो', '-', 'चार', 'साल', 'पर', 'इलाज', 'में', 'महंगाई', 'के', 'मुताबिक़', 'बीमा', 'कंपनियां', 'प्रीमियम', 'में', 'फेरबदल', 'करती', 'हैं', '.']

Token IDs:
[1496, 1457, 410, 521, 16, 617, 694, 410, 1770, 395, 5660, 388, 15121, 4242, 4951, 7365, 395, 13194, 1507, 431, 17]


In [25]:
def evaluate_tokenizer(tokenizer, sentences):

    total_tokens = 0
    total_words = 0
    unknown_tokens = 0

    for sentence in sentences:

        encoding = tokenizer.encode(sentence)

        total_tokens += len(encoding.tokens)
        total_words += len(sentence.split())

        unknown_tokens += encoding.tokens.count("[UNK]")

    avg_tokens_per_sentence = (
        total_tokens / len(sentences)
    )

    avg_tokens_per_word = (
        total_tokens / total_words
    )

    unknown_rate = (
        unknown_tokens / total_tokens
        if total_tokens > 0
        else 0
    )

    return {
        "Average Tokens/Sentence": avg_tokens_per_sentence,
        "Average Tokens/Word": avg_tokens_per_word,
        "UNK Rate": unknown_rate
    }

In [26]:
results_bpe_training = []

VOCAB_SIZE = 20_000

for train_size in TRAINING_SIZES:

    print(
        f"\nTraining BPE with {train_size:,} sentences..."
    )

    tokenizer = train_bpe(
        training_sets[train_size],
        vocab_size=VOCAB_SIZE
    )

    dev_result = evaluate_tokenizer(
        tokenizer,
        dev_sentences
    )

    test_result = evaluate_tokenizer(
        tokenizer,
        test_sentences
    )

    results_bpe_training.append({
        "Tokenizer": "BPE",
        "Training Size": train_size,
        "Vocabulary Size": tokenizer.get_vocab_size(),
        "Dev Avg Tokens/Sentence":
            dev_result["Average Tokens/Sentence"],
        "Dev Avg Tokens/Word":
            dev_result["Average Tokens/Word"],
        "Test Avg Tokens/Sentence":
            test_result["Average Tokens/Sentence"],
        "Test Avg Tokens/Word":
            test_result["Average Tokens/Word"]
    })

bpe_training_df = pd.DataFrame(
    results_bpe_training
)

display(bpe_training_df)


Training BPE with 100,000 sentences...

Training BPE with 300,000 sentences...

Training BPE with 500,000 sentences...

Training BPE with 1,000,000 sentences...


,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,BPE,100000,20000,22.097,1.209734,21.720,1.185331
1,BPE,300000,20000,22.035,1.206340,21.708,1.184676
2,BPE,500000,20000,22.037,1.206449,21.698,1.184130
3,BPE,1000000,20000,22.062,1.207818,21.705,1.184512


In [27]:
results_wp_training = []

VOCAB_SIZE = 20_000

for train_size in TRAINING_SIZES:

    print(
        f"\nTraining WordPiece with {train_size:,} sentences..."
    )

    tokenizer = train_wordpiece(
        training_sets[train_size],
        vocab_size=VOCAB_SIZE
    )

    dev_result = evaluate_tokenizer(
        tokenizer,
        dev_sentences
    )

    test_result = evaluate_tokenizer(
        tokenizer,
        test_sentences
    )

    results_wp_training.append({
        "Tokenizer": "WordPiece",
        "Training Size": train_size,
        "Vocabulary Size": tokenizer.get_vocab_size(),
        "Dev Avg Tokens/Sentence":
            dev_result["Average Tokens/Sentence"],
        "Dev Avg Tokens/Word":
            dev_result["Average Tokens/Word"],
        "Test Avg Tokens/Sentence":
            test_result["Average Tokens/Sentence"],
        "Test Avg Tokens/Word":
            test_result["Average Tokens/Word"]
    })

wp_training_df = pd.DataFrame(
    results_wp_training
)

display(wp_training_df)


Training WordPiece with 100,000 sentences...

Training WordPiece with 300,000 sentences...

Training WordPiece with 500,000 sentences...

Training WordPiece with 1,000,000 sentences...


,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,WordPiece,100000,20000,22.394,1.225994,22.010,1.201157
1,WordPiece,300000,20000,22.338,1.222928,21.997,1.200448
2,WordPiece,500000,20000,22.341,1.223092,22.002,1.200720
3,WordPiece,1000000,20000,22.376,1.225008,22.037,1.202630


In [28]:
training_size_results = pd.concat(
    [
        bpe_training_df,
        wp_training_df
    ],
    ignore_index=True
)

display(training_size_results)

,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,BPE,100000,20000,22.097,1.209734,21.720,1.185331
1,BPE,300000,20000,22.035,1.206340,21.708,1.184676
2,BPE,500000,20000,22.037,1.206449,21.698,1.184130
3,BPE,1000000,20000,22.062,1.207818,21.705,1.184512
4,WordPiece,100000,20000,22.394,1.225994,22.010,1.201157
5,WordPiece,300000,20000,22.338,1.222928,21.997,1.200448
6,WordPiece,500000,20000,22.341,1.223092,22.002,1.200720
7,WordPiece,1000000,20000,22.376,1.225008,22.037,1.202630


In [29]:
VOCAB_SIZES = [
    20_000,
    30_000,
    50_000
]

results_bpe_vocab = []

for vocab_size in VOCAB_SIZES:

    print(
        f"\nTraining BPE with vocabulary {vocab_size:,}..."
    )

    tokenizer = train_bpe(
        training_sets[1_000_000],
        vocab_size=vocab_size
    )

    dev_result = evaluate_tokenizer(
        tokenizer,
        dev_sentences
    )

    test_result = evaluate_tokenizer(
        tokenizer,
        test_sentences
    )

    results_bpe_vocab.append({
        "Tokenizer": "BPE",
        "Training Size": 1_000_000,
        "Vocabulary Size":
            tokenizer.get_vocab_size(),
        "Dev Avg Tokens/Sentence":
            dev_result["Average Tokens/Sentence"],
        "Dev Avg Tokens/Word":
            dev_result["Average Tokens/Word"],
        "Test Avg Tokens/Sentence":
            test_result["Average Tokens/Sentence"],
        "Test Avg Tokens/Word":
            test_result["Average Tokens/Word"]
    })

bpe_vocab_df = pd.DataFrame(
    results_bpe_vocab
)

display(bpe_vocab_df)


Training BPE with vocabulary 20,000...

Training BPE with vocabulary 30,000...

Training BPE with vocabulary 50,000...


,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,BPE,1000000,20000,22.062,1.207818,21.705,1.184512
1,BPE,1000000,30000,21.452,1.174422,21.179,1.155807
2,BPE,1000000,50000,20.956,1.147268,20.751,1.132449


In [30]:
results_wp_vocab = []

for vocab_size in VOCAB_SIZES:

    print(
        f"\nTraining WordPiece with vocabulary {vocab_size:,}..."
    )

    tokenizer = train_wordpiece(
        training_sets[1_000_000],
        vocab_size=vocab_size
    )

    dev_result = evaluate_tokenizer(
        tokenizer,
        dev_sentences
    )

    test_result = evaluate_tokenizer(
        tokenizer,
        test_sentences
    )

    results_wp_vocab.append({
        "Tokenizer": "WordPiece",
        "Training Size": 1_000_000,
        "Vocabulary Size":
            tokenizer.get_vocab_size(),
        "Dev Avg Tokens/Sentence":
            dev_result["Average Tokens/Sentence"],
        "Dev Avg Tokens/Word":
            dev_result["Average Tokens/Word"],
        "Test Avg Tokens/Sentence":
            test_result["Average Tokens/Sentence"],
        "Test Avg Tokens/Word":
            test_result["Average Tokens/Word"]
    })

wp_vocab_df = pd.DataFrame(
    results_wp_vocab
)

display(wp_vocab_df)


Training WordPiece with vocabulary 20,000...

Training WordPiece with vocabulary 30,000...

Training WordPiece with vocabulary 50,000...


,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,WordPiece,1000000,20000,22.376,1.225008,22.037,1.202630
1,WordPiece,1000000,30000,21.674,1.186576,21.378,1.166667
2,WordPiece,1000000,50000,21.091,1.154659,20.873,1.139107


In [31]:
vocab_results = pd.concat(
    [
        bpe_vocab_df,
        wp_vocab_df
    ],
    ignore_index=True
)

display(vocab_results)

,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,BPE,1000000,20000,22.062,1.207818,21.705,1.184512
1,BPE,1000000,30000,21.452,1.174422,21.179,1.155807
2,BPE,1000000,50000,20.956,1.147268,20.751,1.132449
3,WordPiece,1000000,20000,22.376,1.225008,22.037,1.202630
4,WordPiece,1000000,30000,21.674,1.186576,21.378,1.166667
5,WordPiece,1000000,50000,21.091,1.154659,20.873,1.139107


In [32]:
sample_sentences = [
    dev_sentences[0],
    dev_sentences[1],
    dev_sentences[2]
]

comparison_tokenizer_bpe = train_bpe(
    training_sets[1_000_000],
    vocab_size=20_000
)

comparison_tokenizer_wp = train_wordpiece(
    training_sets[1_000_000],
    vocab_size=20_000
)

for i, sentence in enumerate(sample_sentences):

    bpe_tokens = comparison_tokenizer_bpe.encode(
        sentence
    ).tokens

    wp_tokens = comparison_tokenizer_wp.encode(
        sentence
    ).tokens

    print("=" * 80)
    print("Sentence", i + 1)
    print(sentence)

    print("\nBPE:")
    print(bpe_tokens)

    print("\nWordPiece:")
    print(wp_tokens)

Sentence 1
आम तौर पर दो-चार साल पर इलाज में महंगाई के मुताबिक़ बीमा कंपनियां प्रीमियम में फेरबदल करती हैं.

BPE:
['आम', 'तौर', 'पर', 'दो', '-', 'चार', 'साल', 'पर', 'इलाज', 'में', 'महंगाई', 'के', 'मुताबिक़', 'बीमा', 'कंपनियां', 'प्रीमियम', 'में', 'फेरबदल', 'करती', 'हैं', '.']

WordPiece:
['आम', 'तौर', 'पर', 'दो', '-', 'चार', 'साल', 'पर', 'इलाज', 'में', 'महंगाई', 'के', 'मुताबिक़', 'बीमा', 'कंपनियां', 'प्रीमियम', 'में', 'फेरबदल', 'करती', 'हैं', '.']
Sentence 2
नाम - बॉक्स पर चित्रित मकड़ी का जिक्र नहीं करना - बच्चों के लिए थोड़ा डरावना हो सकता है, लेकिन ये वयस्क पेंसिल हैं।

BPE:
['नाम', '-', 'बॉक्स', 'पर', 'चित्र', 'ित', 'मक', 'ड़ी', 'का', 'जिक्र', 'नहीं', 'करना', '-', 'बच्चों', 'के', 'लिए', 'थोड़ा', 'डराव', 'ना', 'हो', 'सकता', 'है', ',', 'लेकिन', 'ये', 'वयस्क', 'पें', 'सिल', 'हैं', '।']

WordPiece:
['नाम', '-', 'बॉक्स', 'पर', 'चित्र', '##ित', 'मक', '##ड़ी', 'का', 'जिक्र', 'नहीं', 'करना', '-', 'बच्चों', 'के', 'लिए', 'थोड़ा', 'डरा', '##वन', '##ा', 'हो', 'सकता', 'है', ',', 'लेकिन', 'ये', '

In [33]:
sample = dev_sentences[0]

print("Original sentence:")
print(sample)

print("\n" + "=" * 80)

for vocab_size in VOCAB_SIZES:

    tokenizer = train_bpe(
        training_sets[1_000_000],
        vocab_size=vocab_size
    )

    tokens = tokenizer.encode(sample).tokens

    print(f"\nBPE Vocabulary = {vocab_size:,}")
    print("Number of tokens:", len(tokens))
    print(tokens)

Original sentence:
आम तौर पर दो-चार साल पर इलाज में महंगाई के मुताबिक़ बीमा कंपनियां प्रीमियम में फेरबदल करती हैं.


BPE Vocabulary = 20,000
Number of tokens: 21
['आम', 'तौर', 'पर', 'दो', '-', 'चार', 'साल', 'पर', 'इलाज', 'में', 'महंगाई', 'के', 'मुताबिक़', 'बीमा', 'कंपनियां', 'प्रीमियम', 'में', 'फेरबदल', 'करती', 'हैं', '.']

BPE Vocabulary = 30,000
Number of tokens: 21
['आम', 'तौर', 'पर', 'दो', '-', 'चार', 'साल', 'पर', 'इलाज', 'में', 'महंगाई', 'के', 'मुताबिक़', 'बीमा', 'कंपनियां', 'प्रीमियम', 'में', 'फेरबदल', 'करती', 'हैं', '.']

BPE Vocabulary = 50,000
Number of tokens: 21
['आम', 'तौर', 'पर', 'दो', '-', 'चार', 'साल', 'पर', 'इलाज', 'में', 'महंगाई', 'के', 'मुताबिक़', 'बीमा', 'कंपनियां', 'प्रीमियम', 'में', 'फेरबदल', 'करती', 'हैं', '.']


In [34]:
training_size_results.to_csv(
    "training_size_results.csv",
    index=False
)

vocab_results.to_csv(
    "vocabulary_size_results.csv",
    index=False
)

print("Results saved successfully.")

Results saved successfully.


In [35]:
print("===== TRAINING SIZE EXPERIMENT =====")
display(training_size_results)

print("\n===== VOCABULARY SIZE EXPERIMENT =====")
display(vocab_results)


===== TRAINING SIZE EXPERIMENT =====


,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,BPE,100000,20000,22.097,1.209734,21.720,1.185331
1,BPE,300000,20000,22.035,1.206340,21.708,1.184676
2,BPE,500000,20000,22.037,1.206449,21.698,1.184130
3,BPE,1000000,20000,22.062,1.207818,21.705,1.184512
4,WordPiece,100000,20000,22.394,1.225994,22.010,1.201157
5,WordPiece,300000,20000,22.338,1.222928,21.997,1.200448
6,WordPiece,500000,20000,22.341,1.223092,22.002,1.200720
7,WordPiece,1000000,20000,22.376,1.225008,22.037,1.202630



===== VOCABULARY SIZE EXPERIMENT =====


,Tokenizer,Training Size,Vocabulary Size,Dev Avg Tokens/Sentence,Dev Avg Tokens/Word,Test Avg Tokens/Sentence,Test Avg Tokens/Word
0,BPE,1000000,20000,22.062,1.207818,21.705,1.184512
1,BPE,1000000,30000,21.452,1.174422,21.179,1.155807
2,BPE,1000000,50000,20.956,1.147268,20.751,1.132449
3,WordPiece,1000000,20000,22.376,1.225008,22.037,1.202630
4,WordPiece,1000000,30000,21.674,1.186576,21.378,1.166667
5,WordPiece,1000000,50000,21.091,1.154659,20.873,1.139107
